In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print("Working dir:", os.getcwd())

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split

# Adjust path below to where your step7_pca.csv is located in your Drive
RAW_PATH = "/content/drive/MyDrive/Colab Notebooks/data/processed/Dataset7.csv"
if not os.path.exists(RAW_PATH):
    # fallback: try current folder
    RAW_PATH = "Dataset7.csv"

df = pd.read_csv(RAW_PATH)
print("Loaded:", RAW_PATH, "shape:", df.shape)
display(df.head())
print("\nColumns:", df.columns.tolist())


In [ ]:
# Cell 2: Prepare features and target (shared)
from sklearn.model_selection import train_test_split

# Ensure target exists
assert 'selling_price' in df.columns, "selling_price not in dataset"

X = df.drop(columns=['selling_price'])
y = df['selling_price']

# If any NaNs remain, fill with median for modeling convenience
X = X.fillna(X.median())

# Train-test split (same for all members to ensure fair comparison)
RANDOM_SEED = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score, mean_absolute_error

# We assume X columns are PCs already (if step7 produced PCs).
# If X contains PC1..PCn, use them directly; otherwise you can re-run PCA here.
pc_cols = X.columns.tolist()
print("Available feature columns (assumed PCs):", pc_cols)

# If the columns are named PC1, PC2... we can vary n_components
max_pcs = min( min(10, X.shape[1]), X.shape[1])  # limit to 10 for speed
results_pca_reg = []

for n_comp in range(1, max_pcs+1):
    # select first n_comp columns
    Xn_train = X_train.iloc[:, :n_comp]
    Xn_test  = X_test.iloc[:, :n_comp]
    lr = LinearRegression()
    lr.fit(Xn_train, y_train)
    yp = lr.predict(Xn_test)
    results_pca_reg.append((n_comp, r2_score(y_test, yp), mean_absolute_error(y_test, yp)))

# Show results
for n, r2v, mae in results_pca_reg:
    print(f"PCs={n} -> R2: {r2v:.3f}, MAE: {mae:.0f}")

# Visualization: explained variance if original pca model present (optional)
# If step7 included explained variance in df metadata this cell won't find it; just plot R2 vs number of PCs
plt.figure(figsize=(6,4))
plt.plot([r[0] for r in results_pca_reg], [r[1] for r in results_pca_reg], marker='o')
plt.xlabel("Number of PCs used")
plt.ylabel("R² Score")
plt.title("PC count vs R² (Linear Regression)")
plt.grid(True)
plt.show()

print("We vary the number of principal components and observe model performance. "
      "If few PCs reach near-maximum R², dimensionality reduction is effective.")
